# A Theoretical Note on Liquidity Risk and Repurchase Agreements

### 1. The Paradigm of Liquidity Risk
In quantitative finance, liquidity risk bifurcates into two distinct but interconnected dimensions: **Market Liquidity Risk** and **Funding Liquidity Risk**. Market liquidity risk denotes the inability to execute large transactions without causing significant adverse price movements. Conversely, funding liquidity risk refers to an institution's inability to settle obligations with immediateality due to a shortfall in liquid cash, irrespective of the firm's overarching solvency.

To manage funding liquidity risk systematically, institutional investors (such as hedge funds, primary dealers, and central banks) rely heavily on collateralized borrowing arrangements. The quintessential instrument in this domain is the Repurchase Agreement (Repo).

### 2. Mechanics of Repurchase Agreements (Repo)
A Repurchase Agreement is fundamentally a short-term collateralized loan structured as a simultaneous sale and future repurchase of securities.

1.  **The Initiation (Near Leg):** The borrower (cash receiver) sells a portfolio of high-quality liquid assets (HQLA), typically sovereign bonds, to the lender (cash provider) at a purchase price, denoted as $P_{0}$.
2.  **The Maturity (Far Leg):** On a pre-specified maturity date, the borrower repurchases the collateral from the lender at a higher repurchase price, denoted as $P_{T}$.

The difference between $P_{T}$ and $P_{0}$ represents the interest paid on the loan. To protect the lender against market liquidity risk if the borrower defaults, the loan is typically subject to a **haircut** (or initial margin). The cash lent is less than the current market value of the collateral.

#### 2.1 Term Structure of Repos
Repos are classified by their duration, $t$, measured in days:
* **Overnight Repo:** A transaction where $t = 1$ business day. This represents the most liquid segment of the money market and serves as a barometer for short-term institutional stress.
* **Term Repo:** A transaction where $t > 1$ days (often 1 week, 1 month, or up to 1 year). Term repos carry a premium over overnight repos due to term liquidity premiums and heightened counterparty risk over the horizon.

### 3. Quantitative Evaluation

#### 3.1 The Repo Rate
The cost of borrowing in a repurchase agreement is expressed as an annualized yield known as the Repo Rate, denoted as $R_{repo}$. Employing the standard money market convention of Actual/360, the relationship between the purchase price, repurchase price, and term is formalized as:

$$R_{repo} = \left( \frac{P_{T} - P_{0}}{P_{0}} \right) \times \left( \frac{360}{t} \right)$$

Where:
* $P_{T}$ is the Repurchase Price.
* $P_{0}$ is the Initial Purchase Price.
* $t$ is the term of the repo in days.

#### 3.2 Collateral Valuation: Clean vs. Dirty Price
Because the underlying collateral frequently comprises coupon-bearing fixed-income securities, precise valuation is critical for calculating the haircut and monitoring margin calls.

Market quotes for bonds are typically provided as the **Clean Price** ($P_{clean}$), which strips out the interest that has accumulated since the last coupon payment. However, the actual cash exchanged in the settlement of the collateral transfer must reflect the **Full Price**, or **Dirty Price** ($P_{dirty}$). 

The formulation is defined by the inclusion of Accrued Interest ($AI$):

$$P_{dirty} = P_{clean} + AI$$

The Accrued Interest is a linear interpolation of the next coupon payment based on the time elapsed since the preceding coupon date:

$$AI = N \times C \times \left( \frac{d_{accrued}}{B} \right)$$

Where:
* $N$ is the Notional (Face) Value of the bond.
* $C$ is the annualized coupon rate.
* $d_{accrued}$ is the number of days elapsed since the last coupon payment.
* $B$ is the day-count basis for the year (e.g., 360, 365, or Actual).

### 4. Computational Implementation in Python

The following script encapsulates the theoretical constructs discussed above into an object-oriented Python architecture. You can execute the cells below to compute the Repo calculations.

In [ ]:
import numpy as np

class RepoQuantPricer:
    """
    A quantitative toolkit for evaluating Repurchase Agreements and Bond Collateral.
    """
    
    @staticmethod
    def calculate_repo_rate(purchase_price: float, repurchase_price: float, term_days: int, year_basis: int = 360) -> float:
        """
        Calculates the annualized Repo Rate using the Actual/YearBasis convention.
        
        Args:
            purchase_price (float): P_0, the initial cash borrowed.
            repurchase_price (float): P_T, the cash returned at maturity.
            term_days (int): The duration of the repo in days.
            year_basis (int): The money market convention (default is 360).
            
        Returns:
            float: The annualized repo rate as a decimal.
        """
        if term_days <= 0:
            raise ValueError("Term days must be strictly positive.")
            
        return ((repurchase_price - purchase_price) / purchase_price) * (year_basis / term_days)

    @staticmethod
    def classify_repo_term(term_days: int) -> str:
        """
        Delineates between an Overnight Repo and a Term Repo based on duration.
        """
        if term_days == 1:
            return "Overnight Repo"
        elif term_days > 1:
            return "Term Repo"
        else:
            raise ValueError("Invalid term length. Term must be >= 1 day.")

    @staticmethod
    def calculate_accrued_interest(face_value: float, coupon_rate: float, days_accrued: int, year_basis: int = 360) -> float:
        """
        Calculates the Accrued Interest (AI) of a collateral bond.
        """
        if days_accrued < 0:
            raise ValueError("Accrued days cannot be negative.")
            
        return face_value * coupon_rate * (days_accrued / year_basis)

    @staticmethod
    def calculate_dirty_price(clean_price: float, accrued_interest: float) -> float:
        """
        Computes the Full (Dirty) Price of the collateral.
        """
        return clean_price + accrued_interest


In [ ]:
# ---------------------------------------------------------
# Scenario 1: Repurchase Agreement Mechanics
# ---------------------------------------------------------
p_0 = 50000000.0   # Purchase Price ($50M)
p_t = 50006500.0   # Repurchase Price
t_days = 1         # Term in days

repo_type = RepoQuantPricer.classify_repo_term(t_days)
repo_rate = RepoQuantPricer.calculate_repo_rate(p_0, p_t, t_days)

print("--- Repurchase Agreement Evaluation ---")
print(f"Structure : {repo_type} ({t_days} Days)")
print(f"Repo Rate : {repo_rate:.4%}\n")

In [ ]:
# ---------------------------------------------------------
# Scenario 2: Collateral Valuation (Clean vs. Dirty Price)
# ---------------------------------------------------------
notional = 100.0      # Standard pricing base
clean_px = 98.250     # Quoted Clean Price
coupon = 0.0425       # 4.25% Annual Coupon
d_accrued = 75        # Days since last coupon

ai = RepoQuantPricer.calculate_accrued_interest(notional, coupon, d_accrued)
dirty_px = RepoQuantPricer.calculate_dirty_price(clean_px, ai)

print("--- Collateral Pricing Evaluation ---")
print(f"Clean Price       : {clean_px:.4f}")
print(f"Accrued Interest  : {ai:.4f}")
print(f"Full (Dirty) Price: {dirty_px:.4f}")